# NER CLI Predict 调试器

本 Notebook 方便以交互方式调试 `ner_cli.py predict` 命令：
- 单文本预测（--text）
- 文件批量预测（--file）
- 切换输出格式（json/text/conll）与置信度阈值（--confidence-threshold）


In [2]:
# 环境与路径设置
from pathlib import Path
import sys, os

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'ner_cli.py').exists() and (path / 'src').exists():
            return path
    return start

project_root = find_project_root(Path.cwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"当前工作目录: {os.getcwd()}")
print(f"项目根目录: {project_root}")

# 切换到项目根目录，确保相对路径一致
os.chdir(project_root)



当前工作目录: /Users/zexho/Documents/python_script/araBertv2/notebooks
项目根目录: /Users/zexho/Documents/python_script/araBertv2


In [3]:
# 封装 predict 命令的便捷调用
from typing import Optional, List
import subprocess
import json

def run_predict_text(model_path: str, text: str, output_format: str = 'json', confidence: float = 0.5) -> Optional[dict]:
    """调用 `ner_cli.py predict --text`，返回解析后的结果（json 格式时）。"""
    cmd = [
        sys.executable, 'ner_cli.py', 'predict',
        '--model-path', model_path,
        '--text', text,
        '--output-format', output_format,
        '--confidence-threshold', str(confidence)
    ]
    print('命令:', ' '.join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print('ERROR:', res.stderr or res.stdout)
        return None
    if output_format == 'json':
        try:
            return json.loads(res.stdout)
        except Exception:
            print(res.stdout)
            return None
    else:
        print(res.stdout)
        return None


def run_predict_file(model_path: str, file_path: str, output_format: str = 'json', confidence: float = 0.5, output_file: Optional[str] = None):
    """调用 `ner_cli.py predict --file`，可选择保存到文件。"""
    cmd = [
        sys.executable, 'ner_cli.py', 'predict',
        '--model-path', model_path,
        '--file', file_path,
        '--output-format', output_format,
        '--confidence-threshold', str(confidence)
    ]
    if output_file:
        cmd.extend(['--output-file', output_file])
    print('命令:', ' '.join(cmd))
    res = subprocess.run(cmd, capture_output=False, text=True)
    if res.returncode != 0:
        print('命令返回码:', res.returncode)



In [4]:
# 示例：单文本预测
# 请根据你的模型路径与示例文本修改
model_path = 'data/ner/models/uae_address_roberta_v1.0'
text = '123 Sheikh Zayed Road, Dubai, UAE'

result = run_predict_text(model_path, text, output_format='json', confidence=0.5)
if result is not None:
    print('\nTokens and labels:')
    for tok, lab in zip(result.get('tokens', []), result.get('labels', [])):
        print(f'{tok}\t{lab}')
    print('\nEntities:')
    for ent in result.get('entities', []):
        print(ent)



命令: /Users/zexho/Documents/python_script/araBertv2/.venv/bin/python ner_cli.py predict --model-path data/ner/models/uae_address_roberta_v1.0 --text 123 Sheikh Zayed Road, Dubai, UAE --output-format json --confidence-threshold 0.5


KeyboardInterrupt: 

In [ ]:
# 示例：批量文件预测
# 文件每行一个文本；可设置输出文件保存结果
model_path = 'data/ner/models/uae_address_roberta_v1.0'
file_path = 'data/ner/data/uae_xml_roberta_base/sample_texts.txt'  # 如无可自建
output_file = None  # 例如: 'data/ner/result/uae_xml_roberta_base/predictions.json'

# 若未先运行上方“便捷调用”单元格，这里做一次最小兜底定义
if 'run_predict_file' not in globals():
    import sys, subprocess
    def run_predict_file(model_path: str, file_path: str, output_format: str = 'json', confidence: float = 0.5, output_file: str = None):
        cmd = [
            sys.executable, 'ner_cli.py', 'predict',
            '--model-path', model_path,
            '--file', file_path,
            '--output-format', output_format,
            '--confidence-threshold', str(confidence)
        ]
        if output_file:
            cmd.extend(['--output-file', output_file])
        print('命令:', ' '.join(cmd))
        res = subprocess.run(cmd, capture_output=False, text=True)
        if res.returncode != 0:
            print('命令返回码:', res.returncode)

run_predict_file(model_path, file_path, output_format='json', confidence=0.5, output_file=output_file)


NameError: name 'run_predict_file' is not defined